# Transform Races Data

1. Read bronze _races_ table
2. Keep only the columns required for analytics (Drop _url_ column).
3. Standardise column names using snake_case (_racename_ -> _race_name_ for example)
4. Rename columns to make them more meaningful (date -> race_date for example)
5. Remove duplicate records
6. Transform values of columns _race_name_ to Title Case
7. Write the transformed data to silver _races_ table.

In [0]:
%run ../00-common/01.environment-config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.races"
silver_table = f"{catalog_name}.{silver_schema}.races"

### Step 1 - Read bronze _races_ table

In [0]:
# Methode 1 that allows options since it's an API call:
# races_df = spark.read.table(bronze_table)

In [0]:
# Method 2 using a method with no additionnal options
races_df = spark.table(bronze_table)

In [0]:
display(races_df)

**2. Keep only the columns required for analytics (Drop _url_ column).**

In [0]:
# races_selected_df = races_df.select(
#     "circuitId",
#     "circuitName",
#     "lat",
#     "long",
#     "locality",
#     "country",
#     "ingestion_timestamp",
#     "source_file"
# )

_AUTRE METHODE POUR KEEP LES BONNES COLONNES : EST PLUS FLEXIBLE CAR PERMET D'AJOUTER DES ALIAS DIRECTEMENT PAR EXEMPLE OU ENCORE DE CHANGER LES VALUERS D'UNE COLONNE!_

In [0]:
from pyspark.sql import functions as F

In [0]:
races_selected_df = races_df.select(
    F.col("season"),
    F.col("round"),
    F.col("raceName"),
    F.col("date"),
    F.col("circuitId"),
    F.col("ingestion_timestamp"),
    F.col("source_file")
)

**3. Standardise column names using snake_case (_circuitId_ -> _circuit_id_ for example) &
4. Rename columns to make them more meaningful (_lat_ -> latitude for example)**

In [0]:
# ANCIENNE VERSION!!
# races_renamed_df = (
#     races_selected_df
#     .withColumnRenamed("circuitId", "circuit_id")
#     .withColumnRenamed("circuitName", "circuit_name")
#     .withColumnRenamed("lat", "latitude")
#     .withColumnRenamed("long","longitude")                       
# )

In [0]:
races_renamed_df = (
    races_selected_df
    .withColumnsRenamed({
        "circuitId": "circuit_id",
        "raceName": "race_name",
        "date": "race_date"
    })                     
)

In [0]:
display(races_renamed_df)

**5. Remove duplicate records**

In [0]:
# races_distinct_df = races_valid_df.distinct()

In [0]:
races_distinct_df = races_renamed_df.dropDuplicates(["season","round"])

In [0]:
display(races_distinct_df)

**6. Transform values of column _race_name_ to Title Case**

In [0]:
races_final_df = (
    races_distinct_df
        .withColumn("race_name", F.initcap(F.col("race_name")))
)

In [0]:
display(races_final_df)

**7. Write the transformed data to silver races table.**

In [0]:
(
    races_final_df
        .write
        .mode("overwrite")
        .format("delta")
        .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))